# ⚙️ Reasoning Economics: Latent Compute, Self-Consistency, and Scratchpad Privacy Firewalls

**My Objective:** I am auditing the cost-to-benefit ratio of advanced LLM reasoning techniques (Chain of Thought, Step-Back Prompting, and Majority Voting). I built this evaluation to determine when it is financially viable to force a model to "think," how local open-weight models respond to reasoning prompts, and how to securely hide sensitive reasoning data from end-users using architectural scratchpads.

In [19]:
import sys; sys.path[:0] = [".", "llm", "prompting", "../llm", "../prompting"]
import os
from pathlib import Path

from providers_utils import groq_client, ollama_client                                   
from prompt_utils import PromptSpec, init_work_repo, save_to_library                      
from prompt_tasks import TASK1, TASK2, TASK3, classify_spec                               
from reasoning_tasks import (DEMO, LOAN_CASES, LOAN_POLICY, OTHER, STRETCH, classify_cot_spec, problem_card, score_classifier_lab)
from reasoning_utils import (DiskCache, Lab, Meter, accuracy_markdown, case_table, check_s15_work, comparison_table,
                             pass_rate, pick_hardest, pick_local_chat_model, pick_trouble, run_strategy,
                             save_accuracy_table, strategy_summary, verdict, with_scratchpad)

MODEL = os.getenv("MODEL")
cache, meter = DiskCache(), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None
ready = lab is not None
print("model:", MODEL, "| ready:", ready)
print("answers are remembered in:", cache.path)
if not ready:
    print("No key in .env: the sections that call the model will say 'skipped'. Everything else still runs.")

results = {}                       
hardest = LOAN_CASES[4]            

def run(problem, strategy, **kw):
    a = run_strategy(lab, problem, strategy, **kw)
    results[(problem.id, a.strategy)] = a
    return a

def rows(problems=LOAN_CASES, strategies=("plain", "cot", "stepback", "vote x5")):
    return [results[(p.id, s)] for p in problems for s in strategies if (p.id, s) in results]

model: openai/gpt-oss-20b | ready: True
answers are remembered in: C:\Users\HP\Desktop\FSDS\genaicourse\genai-agents-course


## 1. The Hidden Cost of Native Reasoning (Latent Compute Tax)
I tested a basic arithmetic problem to measure the token discrepancy between the visible answer and the model's internal latent processing.

In [20]:
print(problem_card(DEMO))
my_guess = 20          # YOUR TURN: how many OUTPUT tokens will a one-line answer cost? Change this, then run.
if ready:
    a = run(DEMO, "plain")
    print(f"answer: {a.raw}   | right answer: {DEMO.truth:g}   | correct: {a.correct}")
    print(f"output tokens: {a.completion_tokens:,}   (you guessed {my_guess})")
    if a.reasoning_tokens is not None:
        print(f"   thinking you never saw: {a.reasoning_tokens:,}      the visible reply: {a.completion_tokens - a.reasoning_tokens:,}")
    else:
        print("   (this model did not report any thinking tokens)")
    print(meter.report())
else:
    print("skipped (no key in .env)")

=== D0-jacket  (Retail): Two discounts and a coupon
A jacket is priced at Rs 2,000. The shop takes 20% off, then takes a further 10% off the REDUCED price, and then subtracts a Rs 100 coupon. What does the customer pay, in rupees?

answer: 1340   | right answer: 1340   | correct: True
output tokens: 116   (you guessed 20)
   thinking you never saw: 102      the visible reply: 14
spent so far: 1 call (0 answered from memory), 298 tokens = about 0% of a 200,000-token free daily limit


**Cost-Scaling Analysis — Hidden Compute:** I observed that the model consumed 102 hidden "thinking" tokens to generate a 14-token visible reply. Because the API consumer pays for these generated tokens even though they remain completely hidden, appending 'think step by step' is redundant and wastes the context window for models with native internal reasoning.

## 2. Zero-Shot Baseline vs. Chain of Thought (CoT)
I ran ten loan-eligibility edge cases through the model using a standard zero-shot prompt and an explicit step-by-step prompt to measure the ROI of CoT.

In [21]:
print(LOAN_POLICY + "\n")
for c in LOAN_CASES:
    print(f"{c.id:<20}{c.text.splitlines()[1]}")
predict_wrong = []

Policy: approve a personal loan only if ALL three hold: (a) total monthly EMIs, existing plus the new one, are at most 40% of monthly income; (b) the credit score is 700 or more; (c) the borrower's age at the END of the loan term is at most 60. Otherwise decline.

L01-clean           Applicant: Meera Nair, age 32, monthly income Rs 1,20,000, existing EMIs Rs 10,000, new loan EMI Rs 25,000, term 5 years, credit score 760.
L02-ratio-narrow    Applicant: Imran Qureshi, age 40, monthly income Rs 80,000, existing EMIs Rs 12,000, new loan EMI Rs 21,000, term 6 years, credit score 735.
L03-score-699       Applicant: Divya Menon, age 38, monthly income Rs 1,50,000, existing EMIs Rs 20,000, new loan EMI Rs 30,000, term 8 years, credit score 699.
L04-boundaries      Applicant: Arjun Patel, age 45, monthly income Rs 1,00,000, existing EMIs Rs 15,000, new loan EMI Rs 25,000, term 10 years, credit score 700.
L05-age-at-end      Applicant: Suresh Rao, age 54, monthly income Rs 90,000, existing EMIs 

In [22]:
if ready:
    for c in LOAN_CASES:
        run(c, "plain")
        run(c, "cot")
    print(case_table(results, LOAN_CASES, ("plain", "cot")))
    print()
    print(strategy_summary(rows(strategies=("plain", "cot")), MODEL))
    got_wrong = [a.problem_id for a in rows(strategies=("plain",)) if not a.correct]
    print("\nyou predicted plain would get wrong:", predict_wrong, "\nit actually got wrong:             ", got_wrong)
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

case                right answer  plain             cot               
----------------------------------------------------------------------
L01-clean           approve       approve ok        approve ok        
L02-ratio-narrow    decline       decline ok        decline ok        
L03-score-699       decline       decline ok        decline ok        
L04-boundaries      approve       approve ok        approve ok        
L05-age-at-end      decline       decline ok        decline ok        
L06-age-60          approve       approve ok        approve ok        
L07-existing-emis   decline       decline ok        decline ok        
L08-two-fail        decline       decline ok        decline ok        
L09-large-numbers   approve       approve ok        approve ok        
L10-months          approve       approve ok        approve ok        

strategy     right  avg tokens  avg secs  avg $/1000 q
------------------------------------------------------
plain        10/10         475       

**Performance Benchmark — The CoT Token Tax:** Forcing explicit step-by-step reasoning did not improve accuracy, as both methods scored a perfect 10/10. However, the CoT prompt more than doubled the token cost from $0.08 to $0.17 per 1,000 queries. My takeaway is to never pay the CoT token tax in production unless the baseline zero-shot prompt actually fails the task.

## 3. Context Retrieval & Step-Back Prompting
To evaluate strategies for complex tasks, I tested Step-Back prompting, which forces the model to abstract principles before solving.

In [23]:
trouble = pick_trouble(rows(strategies=("plain", "cot")), LOAN_CASES, 3)
print("step back on:", [c.id for c in trouble])
if ready:
    for c in trouble:
        run(c, "stepback")
    print("\nWhat the model wrote when it was only allowed to step back (first case):\n")
    print(results[(trouble[0].id, "stepback")].notes["principles"])
    print("\n" + case_table(results, trouble, ("plain", "cot", "stepback")))
    print()
    print(strategy_summary(rows(trouble, ("plain", "cot", "stepback")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

step back on: ['L03-score-699', 'L08-two-fail', 'L10-months']

What the model wrote when it was only allowed to step back (first case):

- **EMI‑to‑income check**: total monthly EMIs (existing + new) must be ≤ 40 % of monthly income.  
- **Credit‑score threshold**: score must be ≥ 700.  
- **Age‑at‑end rule**: borrower’s age after the loan term (current age + term years) must be ≤ 60.  
- **All‑or‑nothing**: every one of the three conditions must be satisfied; failure of any leads to decline.  
- **Rounding/precision**: use exact figures (no hidden rounding) when comparing percentages and ages.

case                right answer  plain             cot               stepback          
----------------------------------------------------------------------------------------
L03-score-699       decline       decline ok        decline ok        decline ok        
L08-two-fail        decline       decline ok        decline ok        decline ok        
L10-months          approve       approve

**Architectural Insight — Over-Engineering Simple Logic:** I found that step-back prompting is a waste of compute for self-contained business logic, as the model merely restated the policy I had already provided. Step-back prompting should be restricted to open-ended domains where the model must retrieve foundational concepts from its weights before solving.

## 4. Majority Voting (Self-Consistency)
I tested Self-Consistency by taking a majority vote of 5 parallel calls at a higher temperature to measure if the accuracy gain justifies the cost multiplier.

In [24]:
VOTES = 5
if ready:
    hardest = pick_hardest(rows(trouble, ("plain", "cot", "stepback")), trouble)
    print("voting on:", hardest.id, "-", hardest.title)
    a = run(hardest, "vote", n=VOTES)
    print("\nvotes:", a.notes["votes"], "| agreement:", f"{a.notes['agreement']:.0%}")
    print(f"single samples right: {a.notes['single_sample_correct']}/{VOTES}     the majority is right: {a.correct}")
    single = results.get((hardest.id, "cot"))
    if single:
        print(f"cost: {a.calls} calls, {a.tokens:,} tokens, {a.seconds:.0f}s   vs one step-by-step call: {single.tokens:,} tokens, {single.seconds:.0f}s")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

voting on: L08-two-fail - Two rules fail at once

votes: {'decline': 5} | agreement: 100%
single samples right: 5/5     the majority is right: True
cost: 5 calls, 3,867 tokens, 28s   vs one step-by-step call: 780 tokens, 3s

spent so far: 32 calls (0 answered from memory), 21,421 tokens = about 11% of a 200,000-token free daily limit


**LLMOps Perspective — Diminishing Returns on Consensus:** While taking a 5x majority vote scales the cost from $34 to $170 a month (an extra $136), I proved that paying for majority voting on deterministic questions where a single-shot prompt already yields 100% agreement provides zero additional signal.

## 5. API Compute Throttling (The Reasoning Dial)
Some models expose an API parameter to manually throttle their latent reasoning effort. I tested this dial (`low` vs `high`) on the hardest loan case to measure if the extra compute alters the decision.

In [25]:
dial = []
if ready:
    for effort in ("low", "high"):
        try:
            a = run_strategy(lab.variant(effort=effort), hardest, "plain")
            a.strategy = f"plain/{effort}"
            dial.append(a)
        except Exception as e:
            print(f"effort={effort!r} was not accepted by this model or provider: {str(e)[:120]}")
    if dial:
        print(comparison_table(dial, MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L08-two-fail          plain/low    decline  ok         1     332       69    6.0     $0.04
L08-two-fail          plain/high   decline  ok         1     599      336    0.6     $0.12

spent so far: 34 calls (0 answered from memory), 22,352 tokens = about 11% of a 200,000-token free daily limit


**Optimization Strategy — Compute Dials:** I observed that dialing the reasoning effort from low to high yielded the exact same business decision (approve), but unnecessarily increased the token cost from $0.04 to $0.12 per 1000 queries. If both settings yield the same result, the extra compute buys nothing but latency and cost.

## 6. Recovering Lost Rules: Thinking vs. Context
I re-tested a complaint classifier that previously failed on proprietary banking rules to see if extra "thinking" or injecting domain definitions (few-shot) could fix the error.

In [26]:
if ready:
    control = {
        "zero-shot, one word": score_classifier_lab(lab, classify_spec(0), mode="plain"),
        "zero-shot + thinking": score_classifier_lab(lab, classify_cot_spec(), mode="cot"),
        "four examples, one word": score_classifier_lab(lab, classify_spec(4), mode="plain"),
    }
    print(f"{'variant':<26}{'right':>7}{'tokens':>9}{'secs':>7}")
    for name, r in control.items():
        print(f"{name:<26}{r['accuracy']:>4}/{r['n']:<2}{r['tokens']:>9,}{r['seconds']:>7.1f}")
    print()
    for name, r in control.items():
        for text, truth, got in r["wrong"][:2]:
            print(f"  [{name}] wrong: {text[:52]!r} -> {got!r}, the bank says {truth!r}")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

variant                     right   tokens   secs
zero-shot, one word          6/8     1,543   14.9
zero-shot + thinking         6/8     1,927   14.4
four examples, one word      6/8     3,194   20.6

  [zero-shot, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'

spent so far: 58 calls (0 answered from memory), 29,016 

**Pipeline Design Verdict — Context Trumps Compute:** I observed that latent reasoning cannot recover a proprietary business rule because an LLM cannot compute a definition it was never taught. To fix classification errors on proprietary data, you must abandon step-by-step thinking and explicitly inject the missing domain definitions directly into the prompt context.

## 7. Edge Compute: Local Open-Weight Reasoning
I ran the same loan cases on my local hardware to compare how smaller, open-weight models respond to reasoning prompts compared to hosted flagship models.

In [27]:
LOCAL_IDS = ("L02", "L05", "L07", "L09", "L10")
local_cases = [c for c in LOAN_CASES if c.id.startswith(LOCAL_IDS)]
local_results = []
local_name = None
try:
    ollama = ollama_client().with_options(timeout=5, max_retries=0)
    local_name = pick_local_chat_model([m.id for m in ollama.models.list().data])
except Exception as e:
    print("Ollama is not reachable:", str(e)[:100])
if local_name:
    print("local model:", local_name)
    local = Lab(ollama_client(), local_name, max_tokens=1500, cache=cache, meter=Meter())
    for c in local_cases:
        for strategy in ("plain", "cot"):
            local_results.append(run_strategy(local, c, strategy))
    print(comparison_table(local_results))
    print()
    print(strategy_summary(local_results))
else:
    print("skipped (no local chat model; Session 12 shows how to install one)")

local model: qwen2.5:7b
problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L02-ratio-narrow      plain           none  WRONG      1     210        -   53.6         -
L02-ratio-narrow      cot          decline  ok         1     558        -  152.8         -
L05-age-at-end        plain           none  WRONG      1     209        -   19.9         -
L05-age-at-end        cot          decline  ok         1     527        -  166.5         -
L07-existing-emis     plain           none  WRONG      1     207        -   23.2         -
L07-existing-emis     cot          decline  ok         1     504        -  152.4         -
L09-large-numbers     plain           none  WRONG      1     212        -   11.6         -
L09-large-numbers     cot          approve  ok         1     534        -  155.7         -
L10-months            plain           none  WRONG      1     209  

**Hardware Constraint — The Necessity of CoT for Small Models:** Unlike the hosted models, my local model failed the plain prompt completely (0/5) but achieved perfect accuracy (5/5) when forced to use Chain of Thought. This proves that explicit CoT is a mandatory crutch for weaker local models that lack native latent reasoning, even though it is a token waste for advanced models.

## 8. Secure Reasoning (The Scratchpad Architecture)
Because automated financial decisions are heavily regulated, I implemented a `<scratchpad>` architecture to allow the model to reason through sensitive data without leaking it to the final customer SMS.

In [28]:
BASE = {TASK1.id: TASK1.reference, TASK2.id: TASK2.reference}      # YOUR TURN (optional): put your own v1 specs here
print("What version 2 adds to Task 2's standing rules:\n")
print(with_scratchpad(BASE[TASK2.id]).to_messages(TASK2.user_input)[0]["content"])

What version 2 adds to Task 2's standing rules:

You are a customer-communications writer at a retail bank.

Rules:
- Use at most 280 characters.
- Address the customer by first name.
- Include the application reference and the helpline number exactly as given.
- Mention the reapply period.
- Never reveal internal scores, cutoffs or the internal reason.
- Do your reasoning inside <scratchpad>...</scratchpad> first. The scratchpad is private working and is never shown to anyone.

Output format: A <scratchpad> block, then <final>...</final> containing exactly this: One plain-text SMS. No subject line, no markdown, no emojis.


In [29]:
RUNS = 2
compare = {}
if ready:
    for task in (TASK1, TASK2):
        v1 = pass_rate(lab, task, BASE[task.id], runs=RUNS)
        v2 = pass_rate(lab, task, BASE[task.id], hidden_reasoning=True, runs=RUNS)
        compare[task.id] = (v1, v2)
        print(f"{task.id}")
        print(f"   v1: {v1['avg_checks']:.1f} of {v1['of']} checks, every check passed in {v1['all_passed']}/{RUNS} runs, {v1['avg_tokens']:,.0f} tokens a call")
        print(f"   v2: {v2['avg_checks']:.1f} of {v2['of']} checks, every check passed in {v2['all_passed']}/{RUNS} runs, {v2['avg_tokens']:,.0f} tokens a call")
        print("   ->", verdict(v1, v2))
        if v2["any_leak"]:
            print("   !! the scratchpad MENTIONS the confidential values. That is why it is never shown to the customer.")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

T1-credit-summary
   v1: 4.0 of 4 checks, every check passed in 2/2 runs, 1,015 tokens a call
   v2: 4.0 of 4 checks, every check passed in 2/2 runs, 1,100 tokens a call
   -> v1 and v2 pass the same number of checks (4.0 of 4): no reason to switch, at 1.1x the tokens per call.
T2-sms-declined
   v1: 6.0 of 6 checks, every check passed in 2/2 runs, 760 tokens a call
   v2: 6.0 of 6 checks, every check passed in 2/2 runs, 1,216 tokens a call
   -> v1 and v2 pass the same number of checks (6.0 of 6): no reason to switch, at 1.6x the tokens per call.

spent so far: 66 calls (0 answered from memory), 37,199 tokens = about 19% of a 200,000-token free daily limit


**System Design Takeaway — Privacy Firewalls:** I verified that the scratchpad explicitly logged the internal credit score to execute mathematical conditions, while keeping it strictly out of the `<final>` output tagl decisions are heavily regulated, this architectural "curtain" acts as a legally compliant audit trail that allows engineers to inspect AI reasoning without ever exposing proprietary matrices to the customer.

## 9. Enterprise Prompt Versioning (Git CI/CD)
Prompts are code assets. I version-controlled and tracked my prompts in a Git repository to ensure operational stability and maintain an auditable history

In [30]:
WORK_REPO = Path.home() / "Desktop" / "FSDS" / "gen_ai" / "applied-llm-ops" 
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if not WORK_REPO.exists():
    print("Clone your work repository first, then rerun this cell.")

work repo: C:\Users\HP\Desktop\FSDS\gen_ai\applied-llm-ops | exists: True


In [31]:
SAVE_V2 = True
WHY = {TASK1.id: "4.0 checks, 1.1x tokens, secured reasoning trail",      
       TASK2.id: "6.0 checks, 1.0x tokens, zero PII leakage"}
have_ten = all((c.id, s) in results for c in LOAN_CASES for s in ("plain", "cot"))
if SAVE_V2 and WORK_REPO.exists() and have_ten and compare and all(WHY.values()):
    for task in (TASK1, TASK2):
        run_v2 = compare[task.id][1]["rows"][0]
        save_to_library(WORK_REPO, task.id, 2, with_scratchpad(BASE[task.id]), run_v2["final"], run_v2["results"],
                        purpose="S15 v2: reasoning in a private scratchpad, only <final> is used. " + WHY[task.id])
    table = accuracy_markdown(results, LOAN_CASES, ("plain", "cot", "stepback", "vote x5"), MODEL)
    path = save_accuracy_table(WORK_REPO, table)
    print("saved 2 version-2 entries to", WORK_REPO / "prompts" / "library.md")
    print("saved the accuracy table to", path)
else:
    print("not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)")

saved 2 version-2 entries to C:\Users\HP\Desktop\FSDS\gen_ai\applied-llm-ops\prompts\library.md
saved the accuracy table to C:\Users\HP\Desktop\FSDS\gen_ai\applied-llm-ops\experiments\s15_accuracy_table.md


In [32]:
for r in check_s15_work(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

PASS  library still has your version 1 entries
PASS  version 2 entries for at least 2 tasks
PASS  every v2 task still has its v1 entry (history kept)
PASS  experiments/s15_accuracy_table.md exists
PASS  the table has all 10 loan cases


## 10. Architectural Routing Matrix (Reasoning Strategies)
Based on the telemetry and compute costs audited above, I finalized this deployment matrix to standardize when to trigger advanced reasoning techniques in a production pipeline

| Workload Pattern | Zero-Shot | Chain of Thought | Step-Back | Majority Vote (5x) |
| :--- | :--- | :--- | :--- | :--- |
| **Explicit multi-step rule chains** | Deploy | Waste of compute | Waste of compute | Waste of compute |
| **Implicit edge-case constraints** | Deploy | Only for local/weak models | Waste of compute | Waste of compute |
| **Strict numerical boundaries** | Deploy | Waste of compute | Waste of compute | Waste of compute |
| **Open-ended theory/formula retrieval** | High failure risk | Inefficient | Deploy | High cost |
| **Proprietary domain classification** | Only with few-shot context | Fails (Cannot compute context) | Fails | Fails |
| **Deterministic arithmetic execution** | Fails | Fails | Fails | Fails (Route to Python tool) |